In [36]:
import json
import pandas as pd

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.expand_frame_repr', False)

## Load ISA-JSON Data

In [37]:
with open("multi-run-isa.json", "r") as f:
# with open("single-run-isa.json", "r") as f:
    data = json.load(f)
    
print(f"Loaded ISA-JSON with {len(data['studies'])} studies")
print(f"Investigation: {data['title']}")

Loaded ISA-JSON with 16 studies
Investigation: Tool Wear Dataset


## Get All Study Variables

This function retrieves all variables (factors) defined for a study.

In [38]:
def get_study_variables(study_idx=0):
    """
    Get all variables (factors) defined for a study.
    
    Args:
        study_idx: Index of the study (0-based)
    
    Returns:
        pandas DataFrame with variable information
    """
    study = data['studies'][study_idx]
    
    variables = []
    for factor in study.get('factors', []):
        factor_name = factor.get('factorName', '')
        factor_type = factor.get('factorType', {})
        
        variables.append({
            'Variable Name': factor_name,
            'Type': factor_type.get('annotationValue', 'Unknown') if isinstance(factor_type, dict) else str(factor_type),
            '@id': factor.get('@id', '')
        })
    
    df = pd.DataFrame(variables)
    
    print(f"\n{'='*100}")
    print(f"STUDY VARIABLES - Study {study_idx + 1}: {study.get('title', 'Unknown')}")
    print(f"{'='*100}")
    print(f"Total variables: {len(variables)}\n")
    if not df.empty:
        print(df[['Variable Name', 'Type']].to_string(index=False))
    else:
        print("No variables found")
    
    return df

## Get Variable Mappings for Study and Run

This function retrieves the factor values (variable mappings) for a specific study and run combination.

In [39]:
def get_variable_mappings(study_idx=0, run_idx=0):
    """
    Get variable mappings (factor values) for a specific study and run.
    
    Args:
        study_idx: Index of the study (0-based)
        run_idx: Index of the run (0-based)
    
    Returns:
        pandas DataFrame with variable mappings
    """
    study = data['studies'][study_idx]
    
    # Build factor lookup: factor @id -> factor name
    factor_lookup = {}
    for factor in study.get('factors', []):
        factor_id = factor.get('@id', '')
        factor_name = factor.get('factorName', '')
        factor_lookup[factor_id] = factor_name
    
    # Get the samples (runs) from materials
    samples = study.get('materials', {}).get('samples', [])
    
    if run_idx >= len(samples):
        print(f"\nRun index {run_idx} out of range. Study has {len(samples)} runs.")
        return pd.DataFrame()
    
    sample = samples[run_idx]
    sample_name = sample.get('name', f'Run {run_idx + 1}')
    
    # Extract factor values
    mappings = []
    for fv in sample.get('factorValues', []):
        category = fv.get('category', {})
        factor_ref = category.get('@id', '') if isinstance(category, dict) else ''
        factor_name = factor_lookup.get(factor_ref, 'Unknown Variable')
        
        value = fv.get('value', '')
        unit = fv.get('unit', {})
        unit_term = ''
        
        if unit and isinstance(unit, dict):
            unit_id = unit.get('@id', '')
            if unit_id:
                resolved_unit = next((u for u in study.get('unitCategories', []) 
                                    if u.get('@id') == unit_id), None)
                if resolved_unit:
                    unit_term = resolved_unit.get('annotationValue', '')
        
        # Format display value
        if unit_term:
            value_display = f"{value} {unit_term}"
        else:
            value_display = str(value)
        
        mappings.append({
            'Variable': factor_name,
            'Value': value_display
        })
    
    df = pd.DataFrame(mappings)
    
    print(f"\n{'='*100}")
    print(f"VARIABLE MAPPINGS - Study {study_idx + 1}: {study.get('title', 'Unknown')}")
    print(f"Run: {sample_name} (index {run_idx})")
    print(f"{'='*100}")
    if not df.empty:
        print(df.to_string(index=False))
    else:
        print("No variable mappings found")
    
    return df

## Example Usage - Study 1

In [40]:
# Get all variables defined for Study 1
df_vars = get_study_variables(0)


STUDY VARIABLES - Study 1: Case 1
Total variables: 6

Variable Name                Type
         Time                Time
Cutting Speed Operating condition
 Depth of Cut Operating condition
         Feed Operating condition
     Material Operating condition
           VB              Damage


In [45]:
# Get variable mappings for Study 1, Run 1 (first run)
df_mappings = get_variable_mappings(0, 5)


VARIABLE MAPPINGS - Study 1: Case 1
Run: Test Setup Characteristics-5 (index 5)
     Variable                                                                                                                                          Value
         Time     D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_01\Settings\time\Case_01_time_run_06.csv
Cutting Speed       D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_01\Settings\CS\Case_01_time_run_06.csv
 Depth of Cut      D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_01\Settings\DOC\Case_01_time_run_06.csv
         Feed     D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_01\Settings\feed\Case_01_time_run_06.csv
     Material D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_01\Settings\material\C

In [ ]:
# Get variable mappings for Study 1, Run 2 (second run)
df_mappings_run2 = get_variable_mappings(1, 0)


VARIABLE MAPPINGS - Study 2: Case 2
Run: Test Setup Characteristics-1 (index 1)
     Variable                                                                                                                                          Value
         Time     D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_02\Settings\time\Case_02_time_run_02.csv
Cutting Speed       D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_02\Settings\CS\Case_02_time_run_02.csv
 Depth of Cut      D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_02\Settings\DOC\Case_02_time_run_02.csv
         Feed     D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_02\Settings\feed\Case_02_time_run_02.csv
     Material D:\DPL\ISA3\Example Implementations\NASA Tool Wear\01) N_studies_with_1_assay_each\Data_mill\Case_02\Settings\material\C

## Analyze Other Studies and Runs

You can analyze any study and run combination by changing the indices:

In [43]:
# Example: Study 2, Run 3
# df_vars_s2 = get_study_variables(1)
# df_mappings_s2_r3 = get_variable_mappings(1, 2)